# 미라벨 데이터 검사 우선순위 산출

독립적인 새 커널에서 실행하세요. **제출 당시 계산 방식을 보존한 추론 경로**입니다. 학습은 정수 Hour, 추론은 분·초 포함 Hour를 사용하므로 운영용으로 사용하기 전에 통일 및 재검증이 필요합니다. 입력 매핑은 원천 행 위치에 의존하며 기본 확인만으로 연결의 정확성이 완전히 보장되지는 않습니다. 미라벨 데이터에는 정답이 없어 성능을 평가할 수 없습니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score

# 저장소 루트 또는 notebooks/에서 실행
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
required = ["cn7_final.csv", "rg3_final.csv", "moldset_labeled.csv"]
missing = [name for name in required if not (DATA / name).is_file()]
if missing:
    raise FileNotFoundError("data/ 에 사용 권한이 있는 입력을 준비하세요: " + ", ".join(missing))
cn7 = pd.read_csv(DATA / "cn7_final.csv")
rg3 = pd.read_csv(DATA / "rg3_final.csv")
raw = pd.read_csv(DATA / "moldset_labeled.csv")
for df in (cn7, rg3):
    df["TimeStamp"] = pd.to_datetime(df["TimeStamp"], errors="raise")
    if not df["_id"].is_unique:
        raise ValueError("학습 데이터의 _id가 중복되었습니다.")
    if df["PassOrFail"].isna().any() or not set(df["PassOrFail"].unique()).issubset({0, 1}):
        raise ValueError("PassOrFail은 결측 없는 0/1이어야 합니다.")
rg3["Side_RH"] = (rg3["Side"] == "RH").astype(int)

process_features = ['Injection_Time', 'Filling_Time', 'Plasticizing_Time', 'Cycle_Time', 'Clamp_Close_Time', 'Cushion_Position', 'Plasticizing_Position', 'Clamp_Open_Position', 'Max_Injection_Speed', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure', 'Max_Back_Pressure', 'Average_Back_Pressure', 'Barrel_Temperature_1', 'Barrel_Temperature_2', 'Barrel_Temperature_3', 'Barrel_Temperature_4', 'Barrel_Temperature_5', 'Barrel_Temperature_6', 'Hopper_Temperature', 'Mold_Temperature_3', 'Mold_Temperature_4']

if not raw["_id"].is_unique:
    raise ValueError("원천 데이터의 _id가 중복되었습니다.")
if not cn7["_id"].isin(raw["_id"]).all():
    raise ValueError("CN7과 원천 데이터의 ID 연결에 누락이 있습니다.")
cn7_raw = cn7[["_id"]].merge(raw[["_id"] + process_features], on="_id", how="left", validate="one_to_one")
# 제출 당시 학습용 시간 정의를 유지합니다. 추론과의 차이는 docs/limitations.md 참고.
cn7["Hour"] = cn7["TimeStamp"].dt.hour
cn7["Hour_sin"] = np.sin(2 * np.pi * cn7["Hour"] / 24)
cn7["Hour_cos"] = np.cos(2 * np.pi * cn7["Hour"] / 24)


In [ ]:
# ==========================================
# CN7 4차 모델링 ④
# 직전 Shot 대비 공정 변화량 생성
# ==========================================

change_cols = [
    "Max_Injection_Speed",
    "Max_Injection_Pressure",
    "Filling_Time",
    "Plasticizing_Time"
]

# cn7 행과 raw 값 결합
temp = cn7[
    ["_id", "TimeStamp"]
].copy()

for col in change_cols:
    temp[f"raw_{col}"] = cn7_raw[col].values

temp["TimeStamp"] = pd.to_datetime(temp["TimeStamp"])


# ------------------------------------------
# 동일 TimeStamp = 동일 shot
# 대표값 하나만 남김
# ------------------------------------------

raw_change_cols = [
    f"raw_{col}" for col in change_cols
]

shot_df = (
    temp
    .sort_values("TimeStamp")
    .groupby("TimeStamp", as_index=False)[raw_change_cols]
    .first()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


# ------------------------------------------
# 직전 shot 대비 변화량
# ------------------------------------------

for col in change_cols:
    shot_df[f"Delta_{col}"] = (
        shot_df[f"raw_{col}"].diff()
    )


delta_cols = [
    f"Delta_{col}" for col in change_cols
]


# 첫 shot은 이전 shot이 없으므로 0
shot_df[delta_cols] = shot_df[delta_cols].fillna(0)


# ------------------------------------------
# 원래 CN7 행에 다시 붙이기
# ------------------------------------------

cn7 = cn7.merge(
    shot_df[["TimeStamp"] + delta_cols],
    on="TimeStamp",
    how="left",
    validate="many_to_one"
)

print(cn7[delta_cols].describe().round(3))
print()
print("Missing:")
print(cn7[delta_cols].isna().sum())

In [ ]:
required = ["moldset_unlabeled_cn7.csv", "moldset_unlabeled_rg3.csv", "unlabeled_data.csv"]
missing = [name for name in required if not (DATA / name).is_file()]
if missing:
    raise FileNotFoundError("추론 입력이 없습니다: " + ", ".join(missing))
unlabeled_cn7 = pd.read_csv(DATA / required[0])
unlabeled_rg3 = pd.read_csv(DATA / required[1])
raw_unlabeled = pd.read_csv(DATA / required[2])
# 기존 결과를 덮어쓰지 않습니다. 다시 실행하려면 새 커널을 사용하세요.
from datetime import datetime
OUT = ROOT / "results" / datetime.now().strftime("prediction_%Y%m%d_%H%M%S_%f")
OUT.mkdir(parents=True, exist_ok=False)
for df in (unlabeled_cn7, unlabeled_rg3):
    idx = df["Unnamed: 0"]
    if idx.isna().any() or not (idx == idx.astype(int)).all():
        raise ValueError("원천 행 위치는 결측 없는 정수여야 합니다.")
    if not idx.between(0, len(raw_unlabeled)-1).all():
        raise ValueError("원천 데이터의 행 위치 범위를 벗어났습니다.")


In [ ]:
# processed → raw mapping
cn7_idx = unlabeled_cn7["Unnamed: 0"].astype(int)
rg3_idx = unlabeled_rg3["Unnamed: 0"].astype(int)

cn7_meta = raw_unlabeled.iloc[cn7_idx].copy().reset_index(drop=True)
rg3_meta = raw_unlabeled.iloc[rg3_idx].copy().reset_index(drop=True)

# 제품 종류 확인
print("===== CN7 PART_NAME =====")
print(cn7_meta["PART_NAME"].value_counts())

print("\n===== RG3 PART_NAME =====")
print(rg3_meta["PART_NAME"].value_counts())

# Side 생성
cn7_meta["Side"] = cn7_meta["PART_NAME"].str.extract(r"\b(LH|RH)\b")
rg3_meta["Side"] = rg3_meta["PART_NAME"].str.extract(r"\b(LH|RH)\b")

print("\n===== SIDE =====")
print("CN7")
print(cn7_meta["Side"].value_counts(dropna=False))

print("\nRG3")
print(rg3_meta["Side"].value_counts(dropna=False))

# TimeStamp
cn7_meta["TimeStamp"] = pd.to_datetime(cn7_meta["TimeStamp"])
rg3_meta["TimeStamp"] = pd.to_datetime(rg3_meta["TimeStamp"])

print("\n===== DATE RANGE =====")
print("CN7:", cn7_meta["TimeStamp"].min(), "~", cn7_meta["TimeStamp"].max())
print("RG3:", rg3_meta["TimeStamp"].min(), "~", rg3_meta["TimeStamp"].max())

print("\n===== MAPPING CHECK =====")
print("CN7 rows:", len(unlabeled_cn7), len(cn7_meta))
print("RG3 rows:", len(unlabeled_rg3), len(rg3_meta))

print("CN7 missing timestamp:", cn7_meta["TimeStamp"].isna().sum())
print("RG3 missing timestamp:", rg3_meta["TimeStamp"].isna().sum())

print("CN7 missing side:", cn7_meta["Side"].isna().sum())
print("RG3 missing side:", rg3_meta["Side"].isna().sum())
for name, meta in [("CN7", cn7_meta), ("RG3", rg3_meta)]:
    if not meta["PART_NAME"].astype(str).str.contains(name, regex=False).all():
        raise ValueError(name + " 제품 매핑이 맞지 않습니다.")
    if meta["Side"].isna().any() or meta["TimeStamp"].isna().any():
        raise ValueError(name + " 메타데이터 연결에 누락이 있습니다.")


In [ ]:
import numpy as np
import pandas as pd

# 시간형 변환
cn7_meta["TimeStamp"] = pd.to_datetime(cn7_meta["TimeStamp"])

# --------------------------------------------------
# 1) Hour 파생변수
# --------------------------------------------------

hour = (
    cn7_meta["TimeStamp"].dt.hour
    + cn7_meta["TimeStamp"].dt.minute / 60
    + cn7_meta["TimeStamp"].dt.second / 3600
)

unlabeled_cn7["Hour_sin"] = np.sin(2 * np.pi * hour / 24)
unlabeled_cn7["Hour_cos"] = np.cos(2 * np.pi * hour / 24)


# --------------------------------------------------
# 2) Delta 계산에 사용할 RAW 변수
# --------------------------------------------------

change_cols = [
    "Max_Injection_Speed",
    "Max_Injection_Pressure",
    "Filling_Time",
    "Plasticizing_Time"
]

# raw 값을 별도 컬럼으로 붙임
temp = cn7_meta[["TimeStamp"] + change_cols].copy()

raw_change_cols = []

for col in change_cols:
    raw_col = f"raw_{col}"
    temp[raw_col] = temp[col]
    raw_change_cols.append(raw_col)


# --------------------------------------------------
# 3) TimeStamp별 하나의 shot으로 변환
# --------------------------------------------------

shot_df = (
    temp
    .sort_values("TimeStamp")
    .groupby("TimeStamp", as_index=False)[raw_change_cols]
    .first()
    .sort_values("TimeStamp")
    .reset_index(drop=True)
)


# --------------------------------------------------
# 4) 직전 shot 대비 변화량
# --------------------------------------------------

for col in change_cols:
    shot_df[f"Delta_{col}"] = (
        shot_df[f"raw_{col}"].diff()
    )

delta_features = [
    "Delta_Max_Injection_Speed",
    "Delta_Max_Injection_Pressure",
    "Delta_Filling_Time",
    "Delta_Plasticizing_Time"
]


# 첫 shot은 이전 shot이 없으므로 Delta = 0
shot_df[delta_features] = shot_df[delta_features].fillna(0)


# --------------------------------------------------
# 5) 제품 행에 Delta 다시 붙이기
# --------------------------------------------------

delta_map = shot_df[
    ["TimeStamp"] + delta_features
]

cn7_delta_unlabeled = (
    cn7_meta[["TimeStamp"]]
    .merge(
        delta_map,
        on="TimeStamp",
        how="left",
        validate="many_to_one"
    )
)

for col in delta_features:
    unlabeled_cn7[col] = cn7_delta_unlabeled[col].values

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
import pandas as pd
import numpy as np

# 최종 feature
final_features_cn7 = (
    process_features
    + ["Hour_sin", "Hour_cos"]
    + delta_features
)

print("최종 feature 수:", len(final_features_cn7))
print(final_features_cn7)

# 결측 확인
print("\n[Labeled missing]")
print(cn7[final_features_cn7].isna().sum().sum())

print("[Unlabeled missing]")
print(unlabeled_cn7[final_features_cn7].isna().sum().sum())

# 최종 학습 데이터
X_train = cn7[final_features_cn7].copy()
y_train = cn7["PassOrFail"].astype(int)

X_test = unlabeled_cn7[final_features_cn7].copy()

print("\nTrain:", X_train.shape)
print("Test :", X_test.shape)
print("Train defects:", y_train.sum())

# 최종 ExtraTrees
final_cn7_model = ExtraTreesClassifier(
    n_estimators=700,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# 전체 labeled 데이터로 최종 학습
final_cn7_model.fit(X_train, y_train)

# 불량 위험도 예측
cn7_risk_score = final_cn7_model.predict_proba(X_test)[:, 1]

print("\nPrediction complete")
print("예측 개수:", len(cn7_risk_score))
print(pd.Series(cn7_risk_score).describe())

In [ ]:
# CN7 최종 예측 결과

cn7_prediction = pd.DataFrame({
    "_id": cn7_meta["_id"].values,
    "TimeStamp": cn7_meta["TimeStamp"].values,
    "PART_NAME": cn7_meta["PART_NAME"].values,
    "Risk_Score": cn7_risk_score
})

# 위험도 높은 순서: 1등이 가장 위험
cn7_prediction["Risk_Rank"] = (
    cn7_prediction["Risk_Score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# 전체 중 위험도 백분위
cn7_prediction["Risk_Percentile"] = (
    cn7_prediction["Risk_Rank"] / len(cn7_prediction)
)

# 검사 우선순위 구간
cn7_prediction["Risk_Level"] = np.select(
    [
        cn7_prediction["Risk_Percentile"] <= 0.05,
        cn7_prediction["Risk_Percentile"] <= 0.20
    ],
    [
        "High",
        "Medium"
    ],
    default="Low"
)

# 위험도 높은 순으로 보기
cn7_prediction_sorted = (
    cn7_prediction
    .sort_values("Risk_Rank")
    .reset_index(drop=True)
)

print(cn7_prediction["Risk_Level"].value_counts())
print()

display(cn7_prediction_sorted.head(20))

In [ ]:
cn7_prediction_sorted.to_csv(
    OUT / "cn7_final_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# RG3 Side 생성
rg3_meta["Side_RH"] = (
    rg3_meta["PART_NAME"]
    .astype(str)
    .str.contains("RH", case=False, na=False)
    .astype(int)
)

unlabeled_rg3["Side_RH"] = rg3_meta["Side_RH"].values

print("===== RG3 SIDE CHECK =====")
print(unlabeled_rg3["Side_RH"].value_counts())
print()
print("missing:", unlabeled_rg3["Side_RH"].isna().sum())
print("rows:", len(unlabeled_rg3))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

# 최종 feature
final_features_rg3 = (
    process_features
    + ["Side_RH"]
)

print("최종 feature 수:", len(final_features_rg3))

print("\n[Labeled missing]")
print(rg3[final_features_rg3].isna().sum().sum())

print("[Unlabeled missing]")
print(unlabeled_rg3[final_features_rg3].isna().sum().sum())

# 학습 / 예측 데이터
X_train_rg3 = rg3[final_features_rg3].copy()
y_train_rg3 = rg3["PassOrFail"].astype(int)

X_test_rg3 = unlabeled_rg3[final_features_rg3].copy()

print("\nTrain:", X_train_rg3.shape)
print("Test :", X_test_rg3.shape)
print("Train defects:", y_train_rg3.sum())

# 최종 RandomForest
final_rg3_model = RandomForestClassifier(
    n_estimators=700,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

final_rg3_model.fit(
    X_train_rg3,
    y_train_rg3
)

rg3_risk_score = final_rg3_model.predict_proba(
    X_test_rg3
)[:, 1]

print("\nPrediction complete")
print("예측 개수:", len(rg3_risk_score))
print(pd.Series(rg3_risk_score).describe())

In [ ]:
rg3_prediction = pd.DataFrame({
    "_id": rg3_meta["_id"].values,
    "TimeStamp": rg3_meta["TimeStamp"].values,
    "PART_NAME": rg3_meta["PART_NAME"].values,
    "Side_RH": unlabeled_rg3["Side_RH"].values,
    "Risk_Score": rg3_risk_score
})

# 위험도 순위
rg3_prediction["Risk_Rank"] = (
    rg3_prediction["Risk_Score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

rg3_prediction["Risk_Percentile"] = (
    rg3_prediction["Risk_Rank"] / len(rg3_prediction)
)

# CN7과 동일하게 구간은 만들어두되,
# RG3에서는 강한 운영 기준으로 주장하지 않음
rg3_prediction["Risk_Level"] = np.select(
    [
        rg3_prediction["Risk_Percentile"] <= 0.05,
        rg3_prediction["Risk_Percentile"] <= 0.20
    ],
    [
        "High",
        "Medium"
    ],
    default="Low"
)

rg3_prediction_sorted = (
    rg3_prediction
    .sort_values("Risk_Rank")
    .reset_index(drop=True)
)

print(rg3_prediction["Risk_Level"].value_counts())

print("\nSide별 Risk Score")
print(
    rg3_prediction
    .groupby("Side_RH")["Risk_Score"]
    .describe()
)

display(rg3_prediction_sorted.head(20))

rg3_prediction_sorted.to_csv(
    OUT / "rg3_final_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
cn7_out = cn7_prediction.copy()
cn7_out["Product"] = "CN7"

rg3_out = rg3_prediction.copy()
rg3_out["Product"] = "RG3"

# 공통 컬럼만 통합
final_predictions = pd.concat([
    cn7_out[
        [
            "_id", "TimeStamp", "PART_NAME", "Product",
            "Risk_Score", "Risk_Rank",
            "Risk_Percentile", "Risk_Level"
        ]
    ],
    rg3_out[
        [
            "_id", "TimeStamp", "PART_NAME", "Product",
            "Risk_Score", "Risk_Rank",
            "Risk_Percentile", "Risk_Level"
        ]
    ]
], ignore_index=True)

print(final_predictions.shape)
print(final_predictions.groupby(
    ["Product", "Risk_Level"]
).size())

print("\n_id 중복:", final_predictions["_id"].duplicated().sum())
print("Risk Score 결측:", final_predictions["Risk_Score"].isna().sum())

final_predictions.to_csv(
    OUT / "final_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

출력은 로컬 results/ 하위에 저장됩니다. CSV를 GitHub에 업로드하지 마세요. 71,180건은 원본 제출본의 추론 건수이며 정답 기반 평가 건수가 아닙니다.